# Figure 6 — Heatmaps of selectivity 

Two heatmaps showing how median enantioselectivity varies across combinations of catalyst class and reagent type:

| Panel | Rows | Columns |
|-------|------|---------|
| **A** | catalyst class | electrophile (imine) type |
| **B** | catalyst class | nucleophile (donor) type |

Reagent types are assigned automatically from SMARTS substructure rules (mutually exclusive, first match wins). Cell color encodes the **median** ee (robust to outliers in sparsely populated cells); the number of reactions *n* is printed in each cell. Cells with fewer than four reactions are masked, as a median over very few points is not meaningful.

## Outputs

- `figure_06_selectivity_heatmaps.pdf` / `.svg` / `.png`

## 1. Imports and style

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':          9,
    'axes.labelsize':    10,
    'axes.titlesize':    10,
    'axes.linewidth':     0.8,
    'figure.dpi':         120,
    'savefig.dpi':        300,
    'savefig.bbox':      'tight',
    'savefig.pad_inches': 0.05,
    'pdf.fonttype':       42,
    'ps.fonttype':        42,
    'svg.fonttype':      'none',
})

## 2. Load data

Required columns: `organocatalyst`, `electrophile`, `nucleophile`, `organocatalyst_class`, `ee`. The `electrophile` column holds imine SMILES and `nucleophile` holds nucleophilic-donor SMILES (enols, enolizable carbonyls, and other Mannich-type donors).

In [ ]:
DATA_PATH = 'Mannich_dataset.csv'  # adjust path as needed
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['organocatalyst', 'electrophile', 'nucleophile',
                       'organocatalyst_class', 'ee']).reset_index(drop=True)
print(f'Reactions: {len(df)}')

## 3. Reagent classification (SMARTS, first-match priority)

Patterns are ordered specific -> general. Nitro groups use a recursive SMARTS matching both neutral and zwitterionic forms. The set of classes follows the "path B" choice: the largest chemically meaningful families are resolved explicitly, while rare miscellaneous structures remain in an `other` bucket kept deliberately small.

In [ ]:
NITRO = '[$([NX3](=O)=O),$([NX3+](=O)[O-])]'

# --- Nucleophilic donors: priority order (specific -> general) ---
NUCLEOPHILE_RULES = [
    ('nitroalkane',      f'[CX4]-{NITRO}'),
    ('pyrazolone',       '[#6]1=[#7][#7][#6](=O)[#6]1'),
    ('maleimide',        'O=C1[NX3]C(=O)CC1'),
    ('rhodanine/thiazolone', 'C1SC(=[S,O])[#7,#6]C1=O'),
    ('cyanoester',       '[NX1]#[CX2][CX4][CX3](=O)[OX2]'),
    ('pyranone',         'O=c1cc[o,c][c,o]c1'),
    ('acylazole',        '[CX3](=O)n1cccn1'),
    ('oxindole',         'O=C1[#7]c2ccccc2C1'),
    ('azlactone',        'O=C1OC=NC1'),
    ('beta_ketoacid',    '[CX3](=O)[CX4][CX3](=O)[OX2H1]'),
    ('beta_ketoester',   '[#6][CX3](=O)[CX4][CX3](=O)[OX2][#6]'),
    ('malonate',         '[#6][OX2][CX3](=O)[CX4][CX3](=O)[OX2][#6]'),
    ('1,3-dicarbonyl',   '[CX3](=O)[CX4][CX3]=O'),
    ('arylacetate_ester','[c][CX4;H1,H2][CX3](=O)[OX2][#6]'),
    ('thioester',        '[CX3](=O)[SX2][#6]'),
    ('lactone',          '[OX2;R][CX3;R]=O'),
    ('aldehyde',         '[CX3;H1](=O)[#6]'),
    ('ketone',           '[#6][CX3](=O)[#6]'),
]

# --- Imine electrophiles: priority order ---
IMINE_RULES = [
    ('quinazolinone',    '[#7]c(=O)[#7]'),
    ('benzoxazinone',    'O=c1[#6,#7][#7,#6,#8]c2ccccc2o1'),
    ('isatin_ketimine',  'O=C1[#7]c2ccccc2C1=[NX2]'),
    ('cyclic_imine',     '[CX3]=[NX2;R]'),
    ('alpha_imino_ester','[NX2]=[CX3][CX3](=O)[OX2]'),
    ('CF3_ketimine',     '[NX2;H0,H1]=[CX3]C(F)(F)F'),
    ('N-sulfonyl',       '[NX2]=[CX3].[NX2]S(=O)(=O)'),
    ('N-carbamate',      '[NX2]=[CX3].[NX2]C(=O)[OX2]'),
    ('N-aryl',           '[NX2]=[CX3].[NX2]-c'),
    ('N-acyl_other',     '[NX2]=[CX3].[NX2]C(=O)'),
]

def compile_rules(rules):
    compiled = []
    for name, smarts in rules:
        if '.' in smarts:
            parts = [Chem.MolFromSmarts(s) for s in smarts.split('.')]
            compiled.append((name, parts, True))
        else:
            compiled.append((name, Chem.MolFromSmarts(smarts), False))
    return compiled

def classify(smiles, compiled_rules, fallback):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return fallback
    for name, patt, is_multi in compiled_rules:
        if is_multi:
            if all(p is not None and mol.HasSubstructMatch(p) for p in patt):
                return name
        else:
            if patt is not None and mol.HasSubstructMatch(patt):
                return name
    return fallback

nuc_compiled = compile_rules(NUCLEOPHILE_RULES)
imi_compiled = compile_rules(IMINE_RULES)

nuc_cache = {s: classify(s, nuc_compiled, 'other') for s in df['nucleophile'].unique()}
imi_cache = {s: classify(s, imi_compiled, 'other') for s in df['electrophile'].unique()}
df['nucleophile_class'] = df['nucleophile'].map(nuc_cache)
df['imine_class'] = df['electrophile'].map(imi_cache)

print('Nucleophile classes:')
print(df['nucleophile_class'].value_counts().to_string())
print('\nImine classes:')
print(df['imine_class'].value_counts().to_string())

## 4. Readable labels

Catalyst class labels are built automatically from the values present in `df['organocatalyst_class']`, with optional overrides for prettier names. Reagent class labels are explicit dictionaries (their keys come from the SMARTS classifier above and are fixed).

In [ ]:
# Optional overrides for prettier axis labels. Keys here should match
# the values in df['organocatalyst_class']. Classes not listed here
# will be shown with their raw name from the data, with underscores
# replaced by spaces.
CATALYST_LABEL_OVERRIDES = {
    'urea_thiourea_guanidine_isothiourea': 'Ureas, thioureas,\nguanidines, isothioureas',
    'proline_pyrrolidine_derivatives':     'Proline and\npyrrolidine derivatives',
    'squaramide':                          'Squaramides',
    'cinchona_alkaloid':               'Cinchona alkaloids',
    'amino_acid':                          'Amino acids',
    'binaphthyl_azepine':                  'Binaphthyl azepines',
    'phosphoric_acid':                     'Phosphoric acids',
    '1,2-diamine':                         '1,2-Diamines',
    'ammonium_betaine':                    'Ammonium betaines',
    'imidazolidinone':                     'Imidazolidinones',
    'other':                               'Other',
}

def prettify(name):
    """Default fallback: replace underscores with spaces and capitalize."""
    s = str(name).replace('_', ' ')
    return s[:1].upper() + s[1:] if s else s

def lookup_label(cls):
    """Try several normalised forms of the class name when looking up overrides,
    so the dictionary works whether the data uses underscores or spaces."""
    candidates = [cls, str(cls).replace(' ', '_'), str(cls).replace('_', ' ')]
    for c in candidates:
        if c in CATALYST_LABEL_OVERRIDES:
            return CATALYST_LABEL_OVERRIDES[c]
    return prettify(cls)

# Build full label map covering every class actually present in the data
CATALYST_LABELS = {
    cls: lookup_label(cls)
    for cls in df['organocatalyst_class'].dropna().unique()
}
NUCLEOPHILE_LABELS = {
    'ketone':            'Ketone',
    'aldehyde':          'Aldehyde',
    'beta_ketoacid':     u'\u03b2-Ketoacid',
    'beta_ketoester':    u'\u03b2-Ketoester',
    'malonate':          'Malonate',
    '1,3-dicarbonyl':    '1,3-Dicarbonyl',
    'oxindole':          'Oxindole',
    'azlactone':         'Azlactone',
    'nitroalkane':       'Nitroalkane',
    'pyrazolone':        'Pyrazolone',
    'maleimide':         'Maleimide',
    'arylacetate_ester': 'Arylacetate ester',
    'rhodanine/thiazolone': 'Rhodanine/thiazolone',
    'cyanoester':        'Cyanoester',
    'pyranone':          'Pyranone',
    'acylazole':         'Acylazole',
    'thioester':         'Thioester',
    'lactone':           'Lactone',
    'other':             'Other',
}
IMINE_LABELS = {
    'N-aryl':            'N-aryl',
    'N-carbamate':       'N-carbamate',
    'N-sulfonyl':        'N-sulfonyl',
    'N-acyl_other':      'N-acyl (other)',
    'alpha_imino_ester': u'\u03b1-Imino ester',
    'CF3_ketimine':      r'CF$_3$ ketimine',
    'cyclic_imine':      'Cyclic imine',
    'isatin_ketimine':   'Isatin ketimine',
    'quinazolinone':     'Quinazolinone',
    'benzoxazinone':     'Benzoxazinone',
    'other':             'Other',
}

## 5. Heatmap helper

Builds a single heatmap of median ee, with per-cell *n* annotations, classes ordered by size, and cells below the minimum count masked in grey.

In [ ]:
def draw_heatmap(ax, reagent_col, col_labels, title, min_n=3,
                 cmap='RdYlGn', vmin=0, vmax=100, vcenter=70):
    """Draw a median-ee heatmap of organocatalyst_class x reagent_col on `ax`."""
    median = df.pivot_table(values='ee', index='organocatalyst_class',
                            columns=reagent_col, aggfunc='median')
    counts = df.pivot_table(values='ee', index='organocatalyst_class',
                            columns=reagent_col, aggfunc='count')

    # Order rows by size; order columns by size but keep 'other' last
    row_order = df['organocatalyst_class'].value_counts().index
    row_order = [r for r in row_order if r in median.index]
    col_order = df[reagent_col].value_counts().index
    col_order = [c for c in col_order if c in median.columns]
    if 'other' in col_order:
        col_order = [c for c in col_order if c != 'other'] + ['other']
    median = median.loc[row_order, col_order]
    counts = counts.loc[row_order, col_order]

    # Mask cells with too few reactions
    mask = counts < min_n
    data = np.ma.masked_where(mask.values, median.values)

    norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad('#EDEDED')  # grey for masked cells

    im = ax.imshow(data, cmap=cmap_obj, norm=norm, aspect='auto')

    # Tick labels
    ax.set_xticks(np.arange(len(col_order)))
    ax.set_yticks(np.arange(len(row_order)))
    ax.set_xticklabels([col_labels.get(c, c) for c in col_order],
                       rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels([CATALYST_LABELS.get(r, r) for r in row_order], fontsize=8)

    # Thin white gridlines between cells
    ax.set_xticks(np.arange(-0.5, len(col_order), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(row_order), 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1.0)
    ax.tick_params(which='minor', length=0)
    ax.tick_params(which='major', length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Per-cell annotation: median ee (large) + n (small)
    for i in range(len(row_order)):
        for j in range(len(col_order)):
            n = counts.values[i, j]
            if np.isnan(n) or n < min_n:
                continue
            ee_val = median.values[i, j]
            # text color: dark on light cells, white on saturated ends
            rgba = cmap_obj(norm(ee_val))
            luminance = 0.299*rgba[0] + 0.587*rgba[1] + 0.114*rgba[2]
            txt_color = 'black' if luminance > 0.5 else 'white'
            ax.text(j, i - 0.13, f'{ee_val:.0f}', ha='center', va='center',
                    fontsize=8.5, fontweight='bold', color=txt_color)
            ax.text(j, i + 0.24, f'n={int(n)}', ha='center', va='center',
                    fontsize=6.5, color=txt_color)

    ax.set_title(title, fontsize=10, pad=8)
    return im

## 6. Assemble the figure

In [ ]:
fig, (ax_A, ax_B) = plt.subplots(
    2, 1, figsize=(7.6, 11.0),
    gridspec_kw={'hspace': 0.55},
)

im = draw_heatmap(ax_A, 'imine_class', IMINE_LABELS,
                  'Catalyst class \u00d7 electrophile', min_n=4)
draw_heatmap(ax_B, 'nucleophile_class', NUCLEOPHILE_LABELS,
             'Catalyst class \u00d7 nucleophile', min_n=4)

# Panel labels
for ax, lab in [(ax_A, 'A'), (ax_B, 'B')]:
    ax.text(-0.02, 1.10, lab, transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='top', ha='right')

# Shared colorbar
cbar = fig.colorbar(im, ax=[ax_A, ax_B], fraction=0.030, pad=0.02)
cbar.set_label('Median ee, %', fontsize=9)
cbar.outline.set_linewidth(0.8)

plt.show()

## 7. Export

In [ ]:
for ext in ('pdf', 'svg', 'png'):
    fig.savefig(f'figure_06_selectivity_heatmaps.{ext}',
                dpi=300 if ext == 'png' else None)